# Aircraft Inpainting Model Endpoint Test

**Model:** YOLOv8s-OBB (aircraft detection) + LaMa (neural inpainting)  
**Endpoint:** `airport-dt-aircraft-inpainting-dev` on Databricks Model Serving (GPU_MEDIUM, scale-to-zero)  
**Purpose:** Remove real-world aircraft from satellite imagery so the 3D view shows only simulated traffic

## Architecture

```
Esri Satellite Tile (256x256 PNG)
        |
        v
+-----------------------------------+
|  Databricks Model Serving         |
|  (GPU_MEDIUM, scale-to-zero)      |
|                                   |
|  1. YOLOv8s-OBB (DOTA)           |  <- Detects aircraft with oriented bounding boxes
|     conf >= 0.15, class "plane"   |
|                                   |
|  2. Mask generation               |  <- OBB polygons dilated outward for clean edges
|     cv2.fillPoly + scale factor   |
|                                   |
|  3. LaMa inpainting              |  <- Fills masked regions with surrounding texture
|     simple-lama-inpainting        |
+-----------------------------------+
        |
        v
Clean tile (256x256 PNG, aircraft removed)
```

## Components

| Component | Description |
|-----------|-------------|
| **Detection** | YOLOv8s-OBB trained on DOTA satellite dataset. Oriented bounding boxes tightly wrap aircraft at any angle |
| **Inpainting** | LaMa (Large Mask Inpainting) fills masked regions with surrounding tarmac/apron texture |
| **Serving** | MLflow pyfunc model (`AircraftInpaintingModel`) registered in Unity Catalog |
| **Cache** | Lakebase PostgreSQL tile cache keyed by z/x/y + source ETag |

In [ ]:
# Setup & Authentication
import base64
import json
import math
import time

import httpx
from IPython.display import display, HTML
from PIL import Image
import io
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- Configuration ---
DATABRICKS_HOST = "fevm-serverless-stable-3n0ihb.cloud.databricks.com"
INPAINTING_ENDPOINT = "airport-dt-aircraft-inpainting-dev"
ESRI_TILE_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
ZOOM = 17

# --- Authentication ---
# Works in both Databricks notebooks and local Jupyter
def get_token():
    try:
        from databricks.sdk import WorkspaceClient
        w = WorkspaceClient()
        return w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    except Exception:
        pass
    try:
        import subprocess
        result = subprocess.run(
            ["databricks", "auth", "token", "--profile", "FEVM_SERVERLESS_STABLE"],
            capture_output=True, text=True, timeout=15,
        )
        if result.returncode == 0:
            return json.loads(result.stdout)["access_token"]
    except Exception:
        pass
    raise RuntimeError("No Databricks auth token available")

TOKEN = get_token()
print(f"Authenticated (token: {len(TOKEN)} chars)")
print(f"Endpoint: https://{DATABRICKS_HOST}/serving-endpoints/{INPAINTING_ENDPOINT}/invocations")

## Endpoint Parameters

### Input Format

| Field | Type | Description |
|-------|------|-------------|
| `image_b64` | string | Base64-encoded satellite tile image (PNG or JPEG, typically 256x256) |

**Payload:**
```json
{
  "dataframe_split": {
    "columns": ["image_b64"],
    "data": [["<base64-encoded-image>"]]
  }
}
```

### Output Format

| Field | Type | Description |
|-------|------|-------------|
| `clean_image_b64` | string | Base64-encoded inpainted PNG (aircraft removed) |
| `aircraft_count` | int | Number of aircraft detected and removed |
| `detections` | string (JSON) | Array of detection bounding boxes with confidence scores |

**Detection format:**
```json
[{"x1": 10, "y1": 20, "x2": 50, "y2": 80, "confidence": 0.85, "class_name": "plane"}]
```

### Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `confidence_threshold` | 0.15 | YOLO detection confidence. Lower = more detections, more false positives |
| `mask_dilation_px` | 10 | Pixels to expand each detection mask for cleaner inpainting edges |
| `device` | auto | `cuda` on GPU endpoints, `cpu` fallback |

In [ ]:
# Airport Selector
#
# Curated terminal coordinates where Esri satellite imagery shows parked aircraft.
# Each verified visually at zoom 17.

TERMINAL_COORDS = {
    "ATL": (33.6407, -84.4277),   # Concourse T
    "FRA": (50.0500, 8.5700),     # Terminal 1
    "HKG": (22.3107, 113.9159),   # Terminal 1
    "JFK": (40.6413, -73.7781),   # Terminal 4
    "LAX": (33.9422, -118.4093),  # TBIT
    "LHR": (51.4703, -0.4601),    # Terminal 5
    "NRT": (35.7721, 140.3929),   # Terminal 1 apron
    "ORD": (41.9742, -87.9073),   # Terminal 5
    "SFO": (37.6197, -122.3836),  # International Terminal
}

ICAO_MAP = {
    "ATL": "KATL", "FRA": "EDDF", "HKG": "VHHH", "JFK": "KJFK",
    "LAX": "KLAX", "LHR": "EGLL", "NRT": "RJAA", "ORD": "KORD", "SFO": "KSFO",
}


def lat_lon_to_tile(lat: float, lon: float, zoom: int) -> tuple[int, int]:
    """Convert lat/lon to tile x, y at a given zoom level."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return x, y


# --- Select Airport ---
SELECTED_AIRPORT = "SFO"  # Change this to test different airports

lat, lon = TERMINAL_COORDS[SELECTED_AIRPORT]
tile_x, tile_y = lat_lon_to_tile(lat, lon, ZOOM)
icao = ICAO_MAP[SELECTED_AIRPORT]

print(f"Airport: {SELECTED_AIRPORT} ({icao})")
print(f"Terminal coords: {lat:.4f}, {lon:.4f}")
print(f"Tile at zoom {ZOOM}: x={tile_x}, y={tile_y}")
print(f"Tile URL: {ESRI_TILE_URL.format(z=ZOOM, y=tile_y, x=tile_x)}")

In [ ]:
# Fetch & Display Satellite Tile

tile_url = ESRI_TILE_URL.format(z=ZOOM, y=tile_y, x=tile_x)

resp = httpx.get(tile_url, timeout=30)
resp.raise_for_status()
original_bytes = resp.content
original_image = Image.open(io.BytesIO(original_bytes))

fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(original_image)
ax.set_title(f"{SELECTED_AIRPORT} ({icao}) - Satellite Tile {ZOOM}/{tile_x}/{tile_y}", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Tile size: {len(original_bytes):,} bytes ({original_image.size[0]}x{original_image.size[1]})")

## Calling the Serving Endpoint

The inpainting model is served via Databricks Model Serving. The endpoint accepts a base64-encoded image and returns the cleaned image with aircraft removed.

**Direct call (Python):**
```python
import base64, httpx

image_b64 = base64.b64encode(tile_bytes).decode()
resp = httpx.post(
    f"https://{HOST}/serving-endpoints/{ENDPOINT}/invocations",
    json={"dataframe_split": {"columns": ["image_b64"], "data": [[image_b64]]}},
    headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},
    timeout=120,
)
result = resp.json()
```

**Via App proxy (curl):**
```bash
curl -X POST \
  "https://airport-digital-twin-dev-*.aws.databricksapps.com/api/inpainting/clean-tile?url=<TILE_URL>&airport_icao=KSFO" \
  -H "Authorization: Bearer $TOKEN" \
  --output cleaned_tile.png
```

**Note:** The endpoint scales to zero after inactivity. First call after cold start takes 2-5 minutes. Warm calls take ~1-5 seconds per tile.

In [ ]:
# Call Inpainting Endpoint

image_b64 = base64.b64encode(original_bytes).decode("utf-8")

serving_url = f"https://{DATABRICKS_HOST}/serving-endpoints/{INPAINTING_ENDPOINT}/invocations"
payload = {
    "dataframe_split": {
        "columns": ["image_b64"],
        "data": [[image_b64]],
    }
}

print(f"Calling {serving_url}")
print(f"Payload size: {len(json.dumps(payload)):,} bytes")

t0 = time.monotonic()
resp = httpx.post(
    serving_url,
    json=payload,
    headers={
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
    },
    timeout=300,  # 5min for cold start
)
elapsed_ms = int((time.monotonic() - t0) * 1000)

print(f"Status: {resp.status_code} ({elapsed_ms}ms)")

if resp.status_code != 200:
    print(f"Error: {resp.text[:500]}")
else:
    result = resp.json()
    predictions = result.get("predictions", result.get("dataframe_split", {}))

    if isinstance(predictions, dict):
        data = predictions.get("data", [[]])[0]
        clean_b64 = data[0] if data else ""
        aircraft_count = data[1] if len(data) > 1 else 0
        detections_json = data[2] if len(data) > 2 else "[]"
    elif isinstance(predictions, list):
        pred = predictions[0]
        clean_b64 = pred.get("clean_image_b64", "")
        aircraft_count = pred.get("aircraft_count", 0)
        detections_json = pred.get("detections", "[]")
    else:
        clean_b64, aircraft_count, detections_json = "", 0, "[]"

    clean_bytes = base64.b64decode(clean_b64)
    detections = json.loads(detections_json) if isinstance(detections_json, str) else detections_json

    print(f"\nAircraft detected: {aircraft_count}")
    print(f"Clean image size: {len(clean_bytes):,} bytes")
    print(f"Detections: {json.dumps(detections, indent=2)}")

In [ ]:
# Display Results: Before vs After with Detection Boxes

clean_image = Image.open(io.BytesIO(clean_bytes))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

# --- Original with detection boxes ---
axes[0].imshow(original_image)
axes[0].set_title(f"Original - {SELECTED_AIRPORT} ({aircraft_count} aircraft detected)", fontsize=11)
for det in detections:
    x1, y1 = det["x1"], det["y1"]
    x2, y2 = det["x2"], det["y2"]
    conf = det.get("confidence", 0)
    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor="red", facecolor="none",
    )
    axes[0].add_patch(rect)
    axes[0].text(
        x1, y1 - 2, f"{conf:.2f}",
        color="red", fontsize=8, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.1", facecolor="white", alpha=0.7),
    )
axes[0].axis("off")

# --- Cleaned image ---
axes[1].imshow(clean_image)
axes[1].set_title(f"Inpainted - Aircraft Removed ({elapsed_ms}ms)", fontsize=11)
axes[1].axis("off")

plt.suptitle(f"Aircraft Inpainting: {SELECTED_AIRPORT} ({icao}) - Tile {ZOOM}/{tile_x}/{tile_y}", fontsize=13)
plt.tight_layout()
plt.show()